In [ ]:
from mp_api.client import MPRester
from pathlib import Path
import json
from datetime import datetime

api_key = ""

PATH = Path.cwd() / "materials_project_reference_data"
PATH.mkdir(parents=True, exist_ok=True)
material = "alpha-3R-In2Se3" 
material_id = "mp-19907"




In [60]:
import json
from pathlib import Path

LOG_FILE = PATH / "mp_pull_log.json"
LOOKUP_FILE = PATH / "phonon_mp_lookup.json"

try:
    with MPRester(api_key) as mpr:
        phonon_doc = mpr.materials.phonon.search(material_id)

        # In addition, you can specfiy the phonon calculation methodology by using `phonon_method`,
        # the default is `None` so all results for mp-19907 are returned. Other options: "dfpt" or "pheasy"
        phonon_doc = mpr.materials.phonon.search(
            material_ids=[material_id], phonon_method="pheasy"
        )
except Exception as e:
    print(f"Error fetching phonon data: {e}")


# Load the existing log (or start a new one)
if LOG_FILE.exists():
    with open(LOG_FILE) as f:
        pull_log = json.load(f)
else:
    pull_log = {}

# Load the existing lookup file
if LOOKUP_FILE.exists():
    with open(LOOKUP_FILE) as f:
        lookup_log = json.load(f)
else:
    lookup_log = {}


# Add / update this material's entry into log file
pull_log[material_id] = {
    "material": material,
    "last_pulled": datetime.now().isoformat(timespec="seconds"),
}

with open(LOG_FILE, "w") as f:
    json.dump(pull_log, f, indent=4)

# Add / update this material's entry into lookup file
lookup_log[material_id] = {
    "material": material,
}
with open(LOOKUP_FILE, "w") as f:
    json.dump(lookup_log, f, indent=4)    
print(phonon_doc)

Output()

Output()

✓ Retrieved 1 SummaryDoc documents in 1.7s

Output()

✓ Retrieved 1 PhononBSDOSTask documents in 0.2s

[MPDataDoc<PhononBSDOSTask>(
primitive_matrix=((0.0, -1.0, 0.0), (-1.0, -0.0, -0.0), (-0.0, 0.0, -1.0)),
nsites=5,
sum_rules_breaking=None,
density=5.762634067297271,
identifier=AlphaID(aaaddfps),
post_process_settings=None,
last_updated=datetime.datetime(2026, 5, 19, 21, 3, 55, 206000, tzinfo=datetime.timezone.utc),
epsilon_static=None,
chemsys='In-Se',
code='vasp',
epsilon_electronic=None,
composition_reduced=Composition('In2 Se3'),
elements=[Element In, Element Se],
supercell_matrix=((2.0, 0.0, 0.0), (0.0, 2.0, 0.0), (0.0, 0.0, 2.0)),
volume=134.42950484614218,
formula_anonymous='A2B3',
formula_units=1,
nelements=2,
density_atomic=26.885900969228437,
calc_meta=[CalcMeta(identifier=None, dir_name='/global/cfs/cdirs/matgen/jzhengap/block_2024-12-27-05-00-07-564806/launcher_2024-12-27-05-02-11-622259/launcher_2025-01-02-04-50-30-922082', name='generate_frequencies_eigenvectors--156157-0', parents=None), CalcMeta(identifier=None, dir_name='/global/cfs/cdirs/matgen/jzhengap/block_2024-12

In [61]:
# Bandstructures
import json

# Set the file save directories
SAVE_PATH = Path.cwd() / "materials_project_reference_data" / "band_structure"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Set the output file name and save location
output_fname = SAVE_PATH / f"{material}-phonondata.json"

# Execute fetch phonon data
try:
    with MPRester(api_key) as mpr:
        # Bandstructure path convention can be specified by using `path_type`. Default is "setyawan_curtarolo".
        phonon_bs = mpr.materials.phonon.get_bandstructure_from_material_id(
        material_id,
            phonon_method="pheasy",
            path_type="latimer_munro",
        )
        if not phonon_bs:
             raise ValueError(f"No phonon data found for material ID: {material_id}")

        phonon_dict = phonon_bs.to_pmg.as_dict()


        with open(output_fname, "w") as json_file:
            json.dump(phonon_dict, json_file, indent=4)
        print(f"Phonon data successfully saved to {output_fname}")

except Exception as e:
    print(f"Error fetching phonon data: {e}")
    


Output()

Phonon data successfully saved to c:\Users\sonny\OneDrive - University of Leeds\year 5\Y5 project\Code\Computational-methods-for-materials-discovery\materials_project_reference_data\band_structure\alpha-3R-In2Se3-phonondata.json


In [62]:
# Force Constants (currently only available for `phonon_method="pheasy"`)
import numpy as np
from phonopy.file_IO import write_FORCE_CONSTANTS

# Set the file save directories
SAVE_PATH = Path.cwd() / "materials_project_reference_data" / "force_constants"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Set the output file name and save location
output_fname = SAVE_PATH / f"{material}_force_constants"

# Execute fetch phonon daya
try:
    with MPRester(api_key) as mpr:
        phonon_force_constants = mpr.materials.phonon.get_forceconstants_from_material_id(
            material_id, phonon_method="pheasy"
        )
    fc = np.array(phonon_force_constants)
    print(fc.shape)
    write_FORCE_CONSTANTS(fc, filename=str(SAVE_PATH / output_fname))  
except Exception as e:
    print(f"Error fetching force constant data {e}")



Output()

(40, 40, 3, 3)


In [63]:
# DOS
# with MPRester(api_key) as mpr:
#    phonon_dos = mpr.materials.phonon.get_dos_from_material_id(
#        material_id, phonon_method="pheasy"
#    )
